In [1]:
import pandas as pd

In [2]:
DF_KB = pd.read_parquet("/home/baptvit/Documents/github/mineracao-dados-massivos/data/med-qa-dataset/medqa_kb")

In [3]:
DF_KB.count()

sentence              861432
embedding_sentence    861432
token_sentence        861432
metadata              861432
dtype: int64

In [4]:
DF_KB.head()

,sentence,embedding_sentence,token_sentence,metadata
0,HCC主要由肝动脉供血，90％以上肿瘤血供丰富；肿瘤压迫周围肝组织可形成假包膜；HCC容易侵...,"[-0.04214775934815407, 0.5282214879989624, 0.4...",127,"Raw file name is: 医学影像学.txt, Were read at: 202..."
1,这种二级－结构是阻止转录继续向下游推进的关键。,"[-0.05957504361867905, 0.22949005663394928, 0....",25,"Raw file name is: all_books.txt, Were read at:..."
2,The holosystolic murmur of chronic TR is gener...,"[-0.20869624614715576, -0.09695304185152054, 0...",98,"Raw file name is: InternalMed_Harrison.txt, We..."
3,Various systemic agents can be used for severe...,"[-0.3980881869792938, 0.006496873218566179, 0....",115,"Raw file name is: InternalMed_Harrison.txt, We..."
4,积极控制高血压是预防心力衰竭发生和防止心力衰竭进展的重要措施。,"[-0.08473484963178635, 0.39388588070869446, -0...",33,"Raw file name is: 临床药理学.txt, Were read at: 202..."


In [5]:
DF_KB.dtypes

sentence              object
embedding_sentence    object
token_sentence         int32
metadata              object
dtype: object

In [11]:
import numpy as np 

DF_KB['embedding_sentence'] = DF_KB['embedding_sentence'].apply(eval).apply(np.array)

SyntaxError: source code string cannot contain null bytes (<string>)

# Evaluation para QA

In [6]:
import pandas as pd

pd_qa_test = pd.read_parquet("/home/baptvit/Documents/github/mineracao-dados-massivos/evaluation/evaluation/data/pd_question_test_embedding_refat_1000")

In [7]:
pd_qa_test.count()

Unnamed: 0                     1000
sentence                       1000
embedding_sentence             1000
token_sentence                 1000
metadata                       1000
rewrited_sentence              1000
rewrited_sentence_embedding    1000
rewrited_sentence_token        1000
dtype: int64

## Pandas Manhattan DIstance

In [8]:
from scipy.spatial import distance as distance_euclidean

def evaluate_top10_manhattan(rewrited_sentence_embedding_value, df=DF_KB):
    def distance_udf(embedding_sentence, rewrited_sentence_embedding):
        return float(distance_euclidean.cityblock(rewrited_sentence_embedding, embedding_sentence))
    df['distances'] = df.apply(lambda row: distance_udf(row['embedding_sentence'], rewrited_sentence_embedding_value), axis=1)
    return df.sort_values(by=['distances']).head(10)

## Manhattan distance

In [9]:
pd_qa = pd_qa_test

In [10]:
import datetime
import time

algo_name = "manhattan_distance" 
algoritimos_name = []
times = []
timestamps = []
top_1_result = []  # If true, the model is write on top 1 else false
top_5_result = (
    []
)  # If true, the model is write on top 5 results else false
top_10_result = (
    []
)  # If true, the model is write on top 10 results else false
distances = []
query_id_result = []

for rows in pd_qa.iterrows():
    algoritimos_name.append(algo_name)
    row = rows[1] 
    query_id = row[0]
    expected_output = row["sentence"]
    rewrited_sentence_embedding = row["rewrited_sentence_embedding"]

    timestamp = datetime.datetime.now()
    timestamps.append(timestamp)
    start_time = time.time()
    #df_top10 = evaluate_top10_euclidean(row)
    df_top10 = evaluate_top10_manhattan(rewrited_sentence_embedding)
    time_query = time.time() - start_time
    times.append(time_query)
    
    #for num, row in enumerate(df_top10):
    for num, rows in enumerate(df_top10.iterrows()):
        row_top_10 = rows[1] 
        # If is first hit
        top_1 = False
        top_5 = False
        top_10 = False
        retrived_sentence = row_top_10["sentence"]
        retrived_distance = row_top_10["distances"]
        if retrived_sentence == expected_output and num == 0:
            top_1 = True
            top_5 = True
            top_10 = True
            distance = retrived_distance
            break
        elif retrived_sentence == expected_output and num <= 5:
            top_1 = False
            top_5 = True
            top_10 = True
            distance = retrived_distance
            break
        elif retrived_sentence == expected_output and num <= 10:
            top_1 = False
            top_5 = False
            top_10 = True
            distance = retrived_distance
            break
        else:
            top_1 = False
            top_5 = False
            top_10 = False
            distance = 1

    distances.append(distance)

    top_1_result.append(top_1)
    top_5_result.append(top_5)
    top_10_result.append(top_10)
    query_id_result.append(query_id)

    msg = f"Query id {query_id},Top 1: {top_1}, Within Top 5 {top_5}, Within Top 10 {top_10} Query time: {time_query}s, Distance {retrived_distance} and with"
    print(msg)

    metrics = {
            "timestamp": [timestamp],
            "algoritimos_name": [algo_name],
            "query_id": [query_id],
            "search_duration": [time_query],
            "top1": [top_1],
            "top5": [top_5],
            "top10": [top_10],
            "distance": [distance],
        }
    df = pd.DataFrame(metrics)
    df.to_csv(f"/home/baptvit/Documents/github/mineracao-dados-massivos/evaluation/evaluation/results/qa_{algo_name}_incremental.csv", mode='a', header=False, index=False)


metrics = {
        "timestamp": timestamps,
        "algoritimos_name": algoritimos_name,
        "query_id": query_id_result,
        "search_duration": times,
        "top1": top_1_result,
        "top5": top_5_result,
        "top10": top_10_result,
        "distance": distances,
    }
df = pd.DataFrame(metrics)
df.to_csv(f"/home/baptvit/Documents/github/mineracao-dados-massivos/evaluation/evaluation/results/qa_{algo_name}_bulk.csv")

/tmp/ipykernel_4566/1266701596.py:21: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  query_id = row[0]


Query id 0,Top 1: True, Within Top 5 True, Within Top 10 True Query time: 5.461705684661865s, Distance 6.444164852247923 and with


/tmp/ipykernel_4566/1266701596.py:21: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  query_id = row[0]


Query id 1,Top 1: True, Within Top 5 True, Within Top 10 True Query time: 5.77344274520874s, Distance 22.473631512686552 and with
Query id 2,Top 1: True, Within Top 5 True, Within Top 10 True Query time: 5.384891986846924s, Distance 23.283399278971046 and with
Query id 3,Top 1: True, Within Top 5 True, Within Top 10 True Query time: 5.497501611709595s, Distance 16.54114331919118 and with
Query id 4,Top 1: True, Within Top 5 True, Within Top 10 True Query time: 5.386209011077881s, Distance 6.14225205067487 and with
Query id 5,Top 1: True, Within Top 5 True, Within Top 10 True Query time: 5.4239342212677s, Distance 27.2619038526027 and with
Query id 6,Top 1: True, Within Top 5 True, Within Top 10 True Query time: 5.481082201004028s, Distance 12.92909930087626 and with
Query id 7,Top 1: True, Within Top 5 True, Within Top 10 True Query time: 5.383768558502197s, Distance 9.275153627764666 and with
Query id 8,Top 1: False, Within Top 5 True, Within Top 10 True Query time: 5.801352024078369s